# CSC3218 — CIFAR-10 CNN (complete assignment, **Google Colab**)

**Standalone notebook** — no `train.py` / `data.py` / `model.py` required. Implements the full pipeline: CIFAR-10 loading, 90/10 train–val split, augmentations, CNN, AdamW + `ReduceLROnPlateau`, early stopping, test metrics, confusion matrix, and sample prediction figures.

**Setup:** `Runtime` → `Change runtime type` → **T4 GPU** (or better). Then **Run all**.

**Outputs:** `/content/csc3218_results/` (Colab) or `./results/` (local). Final cell downloads a zip on Colab.


## 0. Install extra packages (Colab)

PyTorch and matplotlib are already on Colab; this installs/updates lightweight deps only (fast).


In [ ]:
!pip install -q scikit-learn tqdm


## 1. Imports, paths, and device


In [ ]:
from __future__ import annotations

import json
import sys
import time
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from IPython.display import Image, display
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms
from tqdm.auto import tqdm

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    DATA_DIR = "/content/data"
    OUT_DIR = Path("/content/csc3218_results")
else:
    DATA_DIR = "./data"
    OUT_DIR = Path("./results")

OUT_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Colab:", IN_COLAB, "| Device:", device, "| CUDA:", torch.cuda.is_available())
print("Data:", DATA_DIR, "| Out:", OUT_DIR)
print("Torch:", torch.__version__)


## 2. Dataset — CIFAR-10

- **50k** train / **10k** test (official).
- **90% / 10%** train–validation split from the 50k (reproducible seed).
- **Train:** random crop + padding, horizontal flip, normalize.
- **Val / test:** normalize only.


In [ ]:
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)

CIFAR10_CLASSES = (
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
)


def get_transforms(
    augment_train: bool = True,
    use_rotation: bool = False,
) -> tuple[transforms.Compose, transforms.Compose]:
    aug_list: list = [
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
    ]
    if use_rotation:
        aug_list.append(transforms.RandomRotation(degrees=10))
    aug_list.extend(
        [
            transforms.ToTensor(),
            transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
        ]
    )
    train_tf = transforms.Compose(aug_list) if augment_train else transforms.Compose(
        [
            transforms.ToTensor(),
            transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
        ]
    )
    eval_tf = transforms.Compose(
        [
            transforms.ToTensor(),
            transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
        ]
    )
    return train_tf, eval_tf


def get_dataloaders(
    data_dir: str,
    batch_size: int = 64,
    val_fraction: float = 0.1,
    num_workers: int = 2,
    seed: int = 42,
    augment_train: bool = True,
    use_rotation: bool = False,
) -> tuple[DataLoader, DataLoader, DataLoader]:
    train_tf, eval_tf = get_transforms(augment_train=augment_train, use_rotation=use_rotation)

    full_train = datasets.CIFAR10(root=data_dir, train=True, download=True, transform=train_tf)
    test_set = datasets.CIFAR10(root=data_dir, train=False, download=True, transform=eval_tf)

    n_total = len(full_train)
    n_val = int(n_total * val_fraction)
    n_train = n_total - n_val
    generator = torch.Generator().manual_seed(seed)
    train_subset, val_subset = random_split(
        full_train,
        [n_train, n_val],
        generator=generator,
    )

    val_indices = val_subset.indices
    base_train = datasets.CIFAR10(root=data_dir, train=True, download=True, transform=eval_tf)
    val_eval = Subset(base_train, val_indices)

    pin = torch.cuda.is_available()
    nw = 0 if IN_COLAB else num_workers  # Colab: workers often add little on GPU; 0 avoids spawn issues
    train_loader = DataLoader(
        train_subset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=nw,
        pin_memory=pin,
        persistent_workers=(nw > 0),
    )
    val_loader = DataLoader(
        val_eval,
        batch_size=batch_size,
        shuffle=False,
        num_workers=nw,
        pin_memory=pin,
        persistent_workers=(nw > 0),
    )
    test_loader = DataLoader(
        test_set,
        batch_size=batch_size,
        shuffle=False,
        num_workers=nw,
        pin_memory=pin,
        persistent_workers=(nw > 0),
    )
    return train_loader, val_loader, test_loader


## 3. Model — `CIFAR10CNN`

Conv–BN–ReLU stacks, max pooling, dropout, **global average pooling**, FC head → 10 logits (`CrossEntropyLoss`).


In [ ]:
class CIFAR10CNN(nn.Module):
    def __init__(self, num_classes: int = 10, dropout_p: float = 0.3) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(dropout_p * 0.5),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(dropout_p * 0.5),
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(dropout_p),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_p),
            nn.Linear(128, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.gap(x)
        return self.classifier(x)


## 4. Training utilities


In [ ]:
def set_seed(seed: int) -> None:
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    criterion: nn.Module,
) -> tuple[float, float, np.ndarray, np.ndarray]:
    model.eval()
    total_loss = 0.0
    all_preds: list[int] = []
    all_labels: list[int] = []
    n = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        bs = x.size(0)
        total_loss += loss.item() * bs
        n += bs
        preds = logits.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy().tolist())
        all_labels.extend(y.cpu().numpy().tolist())
    avg_loss = total_loss / max(n, 1)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc, np.array(all_labels), np.array(all_preds)


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
) -> tuple[float, float]:
    model.train()
    total_loss = 0.0
    all_preds: list[int] = []
    all_labels: list[int] = []
    n = 0
    for x, y in tqdm(loader, desc="train", leave=False):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        bs = x.size(0)
        total_loss += loss.item() * bs
        n += bs
        preds = logits.argmax(dim=1)
        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_labels.extend(y.cpu().numpy().tolist())
    avg_loss = total_loss / max(n, 1)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc


def plot_curves(history: dict, out_dir: Path) -> None:
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(epochs, history["train_loss"], label="train")
    axes[0].plot(epochs, history["val_loss"], label="validation")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].set_title("Loss curves")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, history["train_acc"], label="train")
    axes[1].plot(epochs, history["val_acc"], label="validation")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].set_title("Accuracy curves")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(out_dir / "curves_loss_accuracy.png", dpi=150)
    plt.close(fig)


def plot_confusion(cm: np.ndarray, class_names: tuple[str, ...], out_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
    ax.figure.colorbar(im, ax=ax)
    ax.set(
        xticks=np.arange(cm.shape[1]),
        yticks=np.arange(cm.shape[0]),
        xticklabels=class_names,
        yticklabels=class_names,
        ylabel="True label",
        xlabel="Predicted label",
        title="Confusion matrix (test set)",
    )
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j, i, format(cm[i, j], "d"),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black",
                fontsize=7,
            )
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


@torch.no_grad()
def plot_sample_predictions(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    out_path: Path,
    n_show: int = 16,
    mean: tuple[float, ...] = CIFAR10_MEAN,
    std: tuple[float, ...] = CIFAR10_STD,
) -> None:
    model.eval()
    images: list[torch.Tensor] = []
    trues: list[int] = []
    preds: list[int] = []
    for x, y in loader:
        x = x.to(device)
        logits = model(x)
        p = logits.argmax(dim=1)
        for i in range(x.size(0)):
            if len(images) >= n_show:
                break
            images.append(x[i].cpu())
            trues.append(int(y[i].item()))
            preds.append(int(p[i].item()))
        if len(images) >= n_show:
            break

    cols = 4
    rows = (n_show + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.2, rows * 2.4))
    axes = np.atleast_2d(axes)
    mean_t = torch.tensor(mean).view(3, 1, 1)
    std_t = torch.tensor(std).view(3, 1, 1)
    for idx in range(rows * cols):
        r, c = divmod(idx, cols)
        ax = axes[r, c]
        if idx < len(images):
            img = images[idx] * std_t + mean_t
            img = img.clamp(0, 1).permute(1, 2, 0).numpy()
            ax.imshow(img)
            ok = trues[idx] == preds[idx]
            color = "green" if ok else "red"
            ax.set_title(
                f"T:{CIFAR10_CLASSES[trues[idx]]}\nP:{CIFAR10_CLASSES[preds[idx]]}",
                fontsize=8, color=color,
            )
        ax.axis("off")
    fig.suptitle("Sample predictions (T=true, P=predicted)", fontsize=11)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


## 5. Hyperparameters

For a **quick test**, lower `EPOCHS` (e.g. `15`) and `PATIENCE` (e.g. `5`). Defaults below target full training (~80 epochs max, early stop at 12 stagnant val-acc epochs).


In [ ]:
SEED = 42
EPOCHS = 80
BATCH_SIZE = 64
LR = 3e-4
WEIGHT_DECAY = 1e-4
DROPOUT = 0.3
PATIENCE = 12
VAL_FRACTION = 0.1
NUM_WORKERS = 2
AUGMENT_TRAIN = True
USE_ROTATION = False

set_seed(SEED)

train_loader, val_loader, test_loader = get_dataloaders(
    data_dir=DATA_DIR,
    batch_size=BATCH_SIZE,
    val_fraction=VAL_FRACTION,
    num_workers=NUM_WORKERS,
    seed=SEED,
    augment_train=AUGMENT_TRAIN,
    use_rotation=USE_ROTATION,
)

xb, yb = next(iter(train_loader))
print("Batch:", tuple(xb.shape), "| Train batches:", len(train_loader))


### Peek at training images (denormalized)


In [ ]:
mean = torch.tensor(CIFAR10_MEAN).view(3, 1, 1)
std = torch.tensor(CIFAR10_STD).view(3, 1, 1)
n_show = min(8, xb.size(0))
fig, axes = plt.subplots(1, n_show, figsize=(2 * n_show, 2.2))
for i in range(n_show):
    img = (xb[i] * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
    ax = axes[i] if n_show > 1 else axes
    ax.imshow(img)
    ax.set_title(CIFAR10_CLASSES[int(yb[i])], fontsize=8)
    ax.axis("off")
plt.suptitle("Sample training batch")
plt.tight_layout()
plt.show()


## 6. Build model and train

**AdamW** + **ReduceLROnPlateau** (val accuracy). **Early stopping** when val accuracy fails to improve for `PATIENCE` epochs. Best checkpoint: `best_model.pt`.


In [ ]:
model = CIFAR10CNN(num_classes=10, dropout_p=DROPOUT).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=4)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")

history = {
    "train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [],
}
best_val_acc = -1.0
best_state = None
epochs_no_improve = 0
start = time.time()

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, device, criterion, optimizer)
    va_loss, va_acc, _, _ = evaluate(model, val_loader, device, criterion)
    scheduler.step(va_acc)

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(va_loss)
    history["train_acc"].append(tr_acc)
    history["val_acc"].append(va_acc)

    print(
        f"Epoch {epoch:03d} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
        f"val loss {va_loss:.4f} acc {va_acc:.4f} | lr {optimizer.param_groups[0]['lr']:.2e}"
    )

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
        torch.save(best_state, OUT_DIR / "best_model.pt")
    else:
        epochs_no_improve += 1

    if epochs_no_improve >= PATIENCE:
        print(f"Early stopping at epoch {epoch} (no val acc improvement for {PATIENCE} epochs).")
        break

if best_state is not None:
    model.load_state_dict(best_state)

elapsed_train = time.time() - start
print(f"\nTraining done in {elapsed_train:.1f}s | best val acc: {best_val_acc:.4f}")


## 7. Final metrics (train / val / test)

Writes `metrics.json` and `classification_report.txt` into the output folder — use for your **report** and **slides**.


In [ ]:
tr_loss, tr_acc, _, _ = evaluate(model, train_loader, device, criterion)
va_loss, va_acc, _, _ = evaluate(model, val_loader, device, criterion)
te_loss, te_acc, y_true, y_pred = evaluate(model, test_loader, device, criterion)

prec_macro = precision_score(y_true, y_pred, average="macro", zero_division=0)
rec_macro = recall_score(y_true, y_pred, average="macro", zero_division=0)
f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)

report = classification_report(
    y_true, y_pred,
    target_names=list(CIFAR10_CLASSES),
    digits=4,
    zero_division=0,
)
cm = confusion_matrix(y_true, y_pred)

hyperparameters = {
    "data_dir": DATA_DIR,
    "out_dir": str(OUT_DIR),
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "dropout": DROPOUT,
    "patience": PATIENCE,
    "val_fraction": VAL_FRACTION,
    "seed": SEED,
    "no_augment": not AUGMENT_TRAIN,
    "rotation": USE_ROTATION,
    "num_workers": NUM_WORKERS,
}

summary = {
    "train_accuracy": float(tr_acc),
    "val_accuracy": float(va_acc),
    "test_accuracy": float(te_acc),
    "test_loss": float(te_loss),
    "test_precision_macro": float(prec_macro),
    "test_recall_macro": float(rec_macro),
    "test_f1_macro": float(f1_macro),
    "epochs_ran": len(history["train_loss"]),
    "seconds": round(elapsed_train, 1),
    "device": str(device),
    "hyperparameters": hyperparameters,
}

with open(OUT_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
with open(OUT_DIR / "classification_report.txt", "w", encoding="utf-8") as f:
    f.write(report)

print("=== Final metrics ===")
print(f"Train acc: {tr_acc:.4f} | Val acc: {va_acc:.4f} | Test acc: {te_acc:.4f}")
print(f"Test loss: {te_loss:.4f}")
print(f"Precision (macro): {prec_macro:.4f} | Recall: {rec_macro:.4f} | F1: {f1_macro:.4f}")
print("\nPer-class report:\n", report)


## 8. Figures

Saves PNGs and shows them below.


In [ ]:
plot_curves(history, OUT_DIR)
plot_confusion(cm, CIFAR10_CLASSES, OUT_DIR / "confusion_matrix_test.png")
plot_sample_predictions(
    model, test_loader, device, OUT_DIR / "sample_predictions.png",
    mean=CIFAR10_MEAN, std=CIFAR10_STD,
)

for name in ("curves_loss_accuracy.png", "confusion_matrix_test.png", "sample_predictions.png"):
    p = OUT_DIR / name
    if p.exists():
        display(Image(filename=str(p)))
    else:
        print("Missing:", p)


## 9. Download results (Colab only)

Zips `csc3218_results/` and triggers a browser download. On a local run, files stay in `./results/`.


In [ ]:
if IN_COLAB:
    zip_path = Path("/content/csc3218_results.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for path in sorted(OUT_DIR.rglob("*")):
            if path.is_file():
                zf.write(path, arcname=Path("csc3218_results") / path.relative_to(OUT_DIR))
    from google.colab import files
    files.download(str(zip_path))
    print("Download started: csc3218_results.zip")
else:
    print("Not on Colab — outputs:", OUT_DIR.resolve())


---

### Report & slides (your work)

- **PDF report:** Architecture, data prep, hyperparameters, final metrics, confusion-matrix discussion. Embed the three PNGs and cite numbers from `metrics.json`.
- **Slides (8–10):** Same storyline, concise.
- **Academic integrity:** Run this notebook yourself; hand in your own figures and text.
